# Module 10 — Count Tokens

Measure a request's **input tokens before sending** — for **cost estimation**, **staying under context limits**, and **comparing prompt variants**.

This builds on **Modules 00–09** and uses the same **`AnthropicVertex` / ADC** path.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging).

In [ ]:
%pip install --quiet "anthropic[vertex]"

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — Count a basic message

`client.messages.count_tokens(model=..., messages=..., system=..., tools=...)` returns an object with **`.input_tokens`**. It does **not** generate anything — it just **measures the input-token cost** of a request, so it's **cheap and fast**.

In [ ]:
count = client.messages.count_tokens(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain what a load balancer does."}],
)
print(f"input_tokens: {count.input_tokens}")

## Part D — Count with system + tools

Counting includes the **system prompt** and **tool definitions** (and image/PDF blocks), so you see the **true input cost** of a full request — not just the user message.

In [ ]:
weather_tool = {
    "name": "get_weather",
    "description": "Get the current weather for a location.",
    "input_schema": {
        "type": "object",
        "properties": {"location": {"type": "string", "description": "City name."}},
        "required": ["location"],
    },
}

count_full = client.messages.count_tokens(
    model=MODEL,
    system="You are a concise assistant. Answer in one short paragraph.",
    tools=[weather_tool],
    messages=[{"role": "user", "content": "Explain what a load balancer does."}],
)
print(f"input_tokens (with system + tools): {count_full.input_tokens}")
print(f"input_tokens (basic, from Part C) : {count.input_tokens}")
print("→ Higher: the system prompt and tool definitions add to the input.")

## Part E — Practical use: compare variants

A common use is **budgeting** — comparing prompt options *before* you spend on a real generation call.

In [ ]:
def count_input_tokens(messages, system=None) -> int:
    kwargs = {"model": MODEL, "messages": messages}
    if system is not None:
        kwargs["system"] = system
    return client.messages.count_tokens(**kwargs).input_tokens

short_prompt = [{"role": "user", "content": "Summarize the benefits of caching."}]
long_prompt = [{"role": "user", "content": (
    "Summarize the benefits of caching. Cover latency, cost, scalability, and "
    "database load, with a concrete example for each, and note one trade-off or "
    "pitfall teams should watch out for when introducing a cache layer."
)}]

print(f"short variant: {count_input_tokens(short_prompt)} input tokens")
print(f"long variant : {count_input_tokens(long_prompt)} input tokens")

## Part F — Notes & recap

### Notes

- **Input only:** `count_tokens` estimates **input** tokens — it does **not** predict output tokens.
- **Separate from `usage`:** this is a pre-flight estimate; the authoritative counts still come from `usage` on a real response.
- **Counts the whole request:** `messages` + `system` + `tools` (+ image/PDF blocks).
- **Great for budgeting:** estimate cost and check you're within context limits before sending.
- Docs: https://docs.claude.com/en/docs/build-with-claude/token-counting and https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/partner-models/claude/count-tokens

### Recap

- `client.messages.count_tokens(...)` returns **`.input_tokens`** without generating anything — cheap and fast.
- It counts the **full request** input: `messages` + `system` + `tools` (+ image/PDF blocks).
- Use it to **estimate cost**, **stay under context limits**, and **compare prompt variants** before spending on a real call.

**Next: Module 11 — Batch predictions.**